In [1]:
#Importing the standard libraries
import numpy as np
np.set_printoptions(legacy='1.25',suppress=True)

import matplotlib.pyplot as plt
%matplotlib qt

#Importing the solver modules
from BaF_solver.system import System
from BaF_solver.obe_with_gradient import obe 
from BaF_solver.states import SigmaLevel,PiLevelParity
from BaF_solver.obe_with_gradient import Excitation, Static_Excitation
import time
import warnings
warnings.filterwarnings('ignore')
import datetime

/Users/mangesh/Documents/Git/BaF-solver/BaF_solver/obe_with_gradient.py:25: UserWarning: Could not detect diffeqpy. Only Python package for OBE solver available.
  warnings.warn(f"Could not detect diffeqpy. Only Python package for OBE solver available.")


In [19]:
#Reference ordering

Bz = 1.0
Ez = 0.0
b=System([0,1],[],B_field = [0.0,0.0,Bz],E_stat_field=[0.0,0.0,Ez],isotope = 138,ignore_mF = False)

start = time.perf_counter()

b.sigma_Hamiltonian.generate_bare()
b.sigma_Hamiltonian.Zeeman.generate_Zeeman()

#Next diagonalize the Hamiltonian for this system
b.sigma_Hamiltonian.diagonalize()


G_global = b.sigma_Hamiltonian.diagonalized_states
GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,3)

print(f"Bz = {Bz}.")
print(f"Static Hamiltonian took : {time.perf_counter()-start}s. ")

G = G_global
GH = GH_global

vec_prev_sigma = b.sigma_Hamiltonian.diagonalized_states_as_vectors
vec_prev_pi = b.pi_Hamiltonian.diagonalized_states_as_vectors

Bz = 1.0.
Static Hamiltonian took : 0.035969333024695516s. 


In [3]:
len(G)

16

In [20]:
GH_list = []
H0_list = []
BR_list = []

from scipy.linalg import block_diag

Bz_list = np.arange(1,6000,30)
for Bz in Bz_list:
    b = System([0,1],[],B_field = [0.0,0.0,Bz],E_stat_field=[0.0,0.0,Ez],isotope = 138,ignore_mF = False)

    start = time.perf_counter()

    b.sigma_Hamiltonian.generate_bare()
    b.sigma_Hamiltonian.Zeeman.generate_Zeeman()

    #Next diagonalize the Hamiltonian for this system
    b.sigma_Hamiltonian.diagonalize()


    G_global = b.sigma_Hamiltonian.diagonalized_states
    GH_global = b.sigma_Hamiltonian.diagonalized_Hamiltonian
    G_temp = G_global
    GH_temp = GH_global
    NG = len(G_temp)

    vec_current_sigma = b.sigma_Hamiltonian.diagonalized_states_as_vectors

    #check the new ground state's overlap with the the previous ground states    permutation_sigma = []
    vec_current_hermit = vec_current_sigma.conj().T
    permutation_sigma = []
    for i in range(NG):
        #check current ith state ground state
        temp = vec_current_hermit@vec_prev_sigma[:,i]
        max_idx = np.argmax(np.abs(temp))
        permutation_sigma.append(max_idx)


    G_new = [G_temp[i] for i in permutation_sigma]
    GH_new_diag = np.diag(GH_temp)[permutation_sigma]
    GH_list.append(GH_new_diag)

    H0 = np.diag(GH_new_diag)
    assert np.allclose(np.imag(H0),np.zeros(H0.shape))
    H0_list.append(H0.astype(np.float64))

    #rearragne the roder of the the vectors recorded as matrices too
    vec_prev_sigma = vec_current_sigma[:,permutation_sigma]
    

In [5]:
NG

16

In [21]:
# Given data
B_vals = np.arange(1,6000,30)
H0_vals = np.array(H0_list)

from BaF_solver.numba_cubicspline import numba_interpolate
# Flatten for fast interpolation
H0_flat = H0_vals.reshape(len(B_vals), -1)
H0     = numba_interpolate(B_vals, H0_flat)


## Visualizing all the energy states within the N = 0 and N = 1 rotational manifold as a function of magnetic field

In [22]:
from BaF_solver.obe_with_gradient import obe

#VISUALIZING ALL THE ENERGY LEVELS
Bz_list = np.arange(4600,4650,0.1)
energy_total = []
for Bz in Bz_list:
    shape = (len(G),len(G))
    H0_temp = obe.get_interp_array(H0,shape,Bz)
    #H0_temp = H0(Bz).reshape((len(G),len(G)))
    H0_temp_diag = np.diag(H0_temp)
    energy_total.append(H0_temp_diag)
plt.plot(Bz_list, energy_total);
plt.xlabel('Magnetic field (G)', fontsize = 14)
plt.ylabel('Energy (MHz)', fontsize = 14)
plt.xticks(fontsize = 14)
plt.yticks(fontsize = 14)
plt.show()



## Visualizing only the states within the magnetic field range where the N = 0 and N = 1 enery levels cross


In [14]:
Bz_list = np.arange(1,50,1)
energy_total = []
for Bz in Bz_list:
    shape = (len(G),len(G))
    H0_temp = obe.get_interp_array(H0,shape,Bz)
    #H0_temp = H0(Bz).reshape((len(G),len(G)))
    H0_temp_diag = np.diag(H0_temp)
    energy_total.append(H0_temp_diag)
plt.plot(Bz_list, energy_total);
plt.xlabel('Magnetic field (G)', fontsize = 14)
plt.ylabel('Energy (MHz)', fontsize = 14)
plt.xticks(fontsize = 14)
plt.yticks(fontsize = 14)
plt.ylim([4300,9600])
plt.xlim([4000,5000])
plt.show()

In [9]:
T00/cmIn2MHz

NameError: name 'T00' is not defined